# Constrained Optimization — Exercises

8 exercises covering Lagrange multipliers, KKT conditions, projected GD, penalty methods, ADMM, and the SVM dual.

| Format            | Description                                  |
| ----------------- | -------------------------------------------- |
| **Problem**       | Markdown cell with task description          |
| **Your Solution** | Code cell with scaffolding                   |
| **Solution**      | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus                |
| ----- | --------- | -------------------- |
| ★     | 1-3       | Core mechanics       |
| ★★    | 4-6       | Deeper theory        |
| ★★★   | 7-8       | AI / ML applications |

### Topic Map

| Topic    | Exercise |
| -------- | -------- |
| Lagrange multipliers | 1 |
| KKT conditions | 2 |
| Simplex projection | 3 |
| SVM dual | 4 |
| Penalty methods | 5 |
| ADMM for Lasso | 6 |
| Portfolio optimization | 7 |
| Constrained RL | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print('  expected:', expected)
        print('  got     :', got)
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def proj_simplex(v):
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u) - 1
    ind = np.arange(len(v)) + 1
    cond = u - cssv / ind > 0
    rho = ind[cond][-1]
    theta = cssv[cond][-1] / rho
    return np.maximum(v - theta, 0)

def soft_threshold(x, threshold):
    return np.sign(x) * np.maximum(np.abs(x) - threshold, 0)

print('Setup complete.')

---

### Exercise 1: Lagrange Multipliers for Equality Constraints ★

Solve $\min \frac{1}{2}(x_1^2 + x_2^2)$ subject to $x_1 + x_2 = 1$.

**(a)** Form the Lagrangian and derive the KKT conditions.

**(b)** Solve for $\mathbf{x}^*$ and $\nu^*$.

**(c)** Verify geometrically that $\nabla f(\mathbf{x}^*)$ is parallel to the constraint gradient.

In [ ]:
# Exercise 1: Your Solution
A = np.array([[1.0, 1.0]])
b = np.array([1.0])

# Solve via KKT system: [I A^T; A 0] [x; nu] = [0; b]
K = np.block([[np.eye(2), A.T], [A, np.zeros((1, 1))]])
rhs = np.concatenate([np.zeros(2), b])
sol = la.solve(K, rhs)
x_star = sol[:2]
nu_star = sol[2]

print(f'x* = {x_star}')
print(f'nu* = {nu_star:.4f}')

In [ ]:
# Exercise 1: Solution
A = np.array([[1.0, 1.0]])
b = np.array([1.0])
K = np.block([[np.eye(2), A.T], [A, np.zeros((1, 1))]])
rhs = np.concatenate([np.zeros(2), b])
sol = la.solve(K, rhs)
x_star = sol[:2]
nu_star = sol[2]

header('Exercise 1: Lagrange Multipliers')
print(f'x* = {x_star}')
print(f'nu* = {nu_star:.4f}')

check_close('solution is (0.5, 0.5)', x_star, np.array([0.5, 0.5]))
check_close('constraint satisfied', A @ x_star, b)

# Geometric: gradient of f = x* = (0.5, 0.5), constraint gradient = (1, 1)
grad_f = x_star
grad_h = A[0]
print(f'\nGradient of f at x*: {grad_f}')
print(f'Gradient of h: {grad_h}')
print(f'nu* * grad_h = {nu_star * grad_h}')
check_close('grad_f = -nu * grad_h', grad_f, -nu_star * grad_h)

print('\nTakeaway: At the optimum, the objective gradient is balanced by the constraint gradient.')

---

### Exercise 2: KKT Conditions ★

For $\min x_1^2 + x_2^2$ subject to $x_1 + x_2 \geq 1$ and $x_1 \geq 0$, write and solve the KKT conditions.

**(a)** Write the Lagrangian and all four KKT conditions.

**(b)** Solve by case analysis on which constraints are active.

**(c)** Verify dual feasibility ($\lambda_i \geq 0$).

In [ ]:
# Exercise 2: Your Solution
# Problem: min x1^2 + x2^2 s.t. 1-x1-x2 <= 0, -x1 <= 0
# Case analysis:
# Case 2 (only g1 active): x1=x2=0.5, lambda1=1, lambda2=0

x_star = np.array([0.5, 0.5])
lam = np.array([1.0, 0.0])

print(f'x* = {x_star}')
print(f'lambda = {lam}')
print(f'g1(x*) = {1 - x_star[0] - x_star[1]:.6f}')
print(f'g2(x*) = {-x_star[0]:.6f}')

In [ ]:
# Exercise 2: Solution
x_star = np.array([0.5, 0.5])
lam = np.array([1.0, 0.0])

header('Exercise 2: KKT Conditions')
print(f'x* = {x_star}, lambda = {lam}')

# Stationarity
grad_f = 2 * x_star
grad_g1 = np.array([-1.0, -1.0])
grad_g2 = np.array([-1.0, 0.0])
stationarity = grad_f + lam[0]*grad_g1 + lam[1]*grad_g2
check_close('stationarity', stationarity, np.zeros(2), tol=1e-10)

# Primal feasibility
check_true('g1(x*) <= 0', 1 - x_star[0] - x_star[1] <= 1e-10)
check_true('g2(x*) <= 0', -x_star[0] <= 1e-10)

# Dual feasibility
check_true('lambda >= 0', np.all(lam >= -1e-10))

# Complementary slackness
cs1 = lam[0] * (1 - x_star[0] - x_star[1])
cs2 = lam[1] * (-x_star[0])
check_close('complementary slackness 1', cs1, 0.0)
check_close('complementary slackness 2', cs2, 0.0)

print('\nTakeaway: KKT conditions systematically identify the constrained optimum.')

---

### Exercise 3: Projection onto the Simplex ★

Implement the $O(n \log n)$ simplex projection algorithm.

**(a)** Implement the sorting-based algorithm from Duchi et al. (2008).

**(b)** Verify that the result satisfies $\mathbf{1}^\top \mathbf{x} = 1$ and $\mathbf{x} \geq \mathbf{0}$.

**(c)** Test with vectors that are already on the simplex, outside, and with negative entries.

In [ ]:
# Exercise 3: Your Solution
def proj_simplex_impl(v):
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u) - 1
    ind = np.arange(len(v)) + 1
    cond = u - cssv / ind > 0
    rho = ind[cond][-1]
    theta = cssv[cond][-1] / rho
    return np.maximum(v - theta, 0)

# Test cases
v1 = np.array([0.2, 0.3, 0.5])
v2 = np.array([1.0, 2.0, 3.0])
v3 = np.array([0.5, -0.3, 0.8])

for i, v in enumerate([v1, v2, v3], 1):
    p = proj_simplex_impl(v)
    print(f'Test {i}: sum={p.sum():.6f}, min={p.min():.6f}')


In [ ]:
# Exercise 3: Solution
def proj_simplex_impl(v):
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u) - 1
    ind = np.arange(len(v)) + 1
    cond = u - cssv / ind > 0
    rho = ind[cond][-1]
    theta = cssv[cond][-1] / rho
    return np.maximum(v - theta, 0)

v1 = np.array([0.2, 0.3, 0.5])
v2 = np.array([1.0, 2.0, 3.0])
v3 = np.array([0.5, -0.3, 0.8])

header('Exercise 3: Simplex Projection')
for i, v in enumerate([v1, v2, v3], 1):
    p = proj_simplex_impl(v)
    print(f'Test {i}: input={v}')
    print(f'  projected={p}, sum={p.sum():.10f}, min={p.min():.10f}')

p1 = proj_simplex_impl(v1)
p2 = proj_simplex_impl(v2)
p3 = proj_simplex_impl(v3)

check_close('already on simplex: unchanged', p1, v1)
check_close('sum = 1 (test 2)', p2.sum(), 1.0)
check_close('sum = 1 (test 3)', p3.sum(), 1.0)
check_true('non-negative (test 2)', p2.min() >= -1e-10)
check_true('non-negative (test 3)', p3.min() >= -1e-10)

print('\nTakeaway: Simplex projection is essential for probability distributions and attention weights.')

---

### Exercise 4: SVM Dual Derivation ★★

Derive and solve the soft-margin SVM dual problem.

**(a)** Write the Lagrangian and derive the dual problem.

**(b)** Show that $0 \leq \alpha_i \leq C$ and $\sum \alpha_i y_i = 0$.

**(c)** Solve the dual for a small dataset and recover $\mathbf{w}$ and $b$.

In [ ]:
# Exercise 4: Your Solution
from scipy.optimize import minimize
X = np.array([[2, 2], [3, 3], [-1, -1], [-2, -2]])
y = np.array([1, 1, -1, -1])
C = 1.0
n = len(y)
K = np.outer(y, y) * (X @ X.T)

def dual_obj(alpha):
    return 0.5 * alpha @ K @ alpha - np.sum(alpha)

bounds = [(0, C) for _ in range(n)]
constraints = {'type': 'eq', 'fun': lambda a: np.sum(a * y)}
res = minimize(dual_obj, np.zeros(n), bounds=bounds, constraints=constraints)
alpha = res.x
sv_idx = alpha > 1e-5
w = np.sum(alpha[sv_idx, None] * y[sv_idx, None] * X[sv_idx], axis=0)
print(f'alpha = {alpha}')
print(f'Support vectors: {np.sum(sv_idx)}')
print(f'w = {w}')

In [ ]:
# Exercise 4: Solution
from scipy.optimize import minimize
X = np.array([[2, 2], [3, 3], [-1, -1], [-2, -2]])
y = np.array([1, 1, -1, -1])
C = 1.0
n = len(y)
K = np.outer(y, y) * (X @ X.T)

def dual_obj(alpha):
    return 0.5 * alpha @ K @ alpha - np.sum(alpha)

bounds = [(0, C) for _ in range(n)]
constraints = {'type': 'eq', 'fun': lambda a: np.sum(a * y)}
res = minimize(dual_obj, np.zeros(n), bounds=bounds, constraints=constraints)
alpha = res.x
sv_idx = alpha > 1e-5
w = np.sum(alpha[sv_idx, None] * y[sv_idx, None] * X[sv_idx], axis=0)
margin_sv = (alpha > 1e-5) & (alpha < C - 1e-5)
b = np.mean(y[margin_sv] - X[margin_sv] @ w) if np.any(margin_sv) else 0

header('Exercise 4: SVM Dual')
print(f'alpha = {alpha}')
print(f'Support vectors: {np.sum(sv_idx)}')
print(f'w = {w}, b = {b:.6f}')

check_true('0 <= alpha <= C', np.all(alpha >= -1e-10) and np.all(alpha <= C + 1e-10))
check_close('sum(alpha*y) = 0', np.sum(alpha * y), 0.0)
check_true('at least one support vector', np.sum(sv_idx) > 0)

margins = y * (X @ w + b)
print(f'\nMargins: {margins}')
check_true('all margins >= 1 - tolerance', np.all(margins >= 1 - 1e-4))

print('\nTakeaway: The SVM dual reveals the kernel trick and the sparsity of support vectors.')

---

### Exercise 5: Penalty Method Convergence ★★

Apply the quadratic penalty method to $\min x^2$ subject to $x \geq 1$.

**(a)** Write the penalized objective $P(x, \mu) = x^2 + \mu \max(0, 1-x)^2$.

**(b)** Solve analytically for the minimizer $x(\mu)$.

**(c)** Show that $x(\mu) \to 1$ as $\mu \to \infty$.

In [ ]:
# Exercise 5: Your Solution
from scipy.optimize import minimize_scalar

def f_pen(x, mu):
    return x**2 + mu * max(0, 1 - x)**2

mu_values = [1, 10, 100, 1000, 10000]
results = []
for mu in mu_values:
    res = minimize_scalar(lambda x: f_pen(x, mu), bounds=(-5, 5), method='bounded')
    results.append((mu, res.x, max(0, 1-res.x)))

print(f"{'mu':>8} | {'x(mu)':>10} | {'Violation':>10}")
print('-' * 35)
for mu, x_mu, viol in results:
    print(f'{mu:>8} | {x_mu:>10.6f} | {viol:>10.6f}')

In [ ]:
# Exercise 5: Solution
from scipy.optimize import minimize_scalar

def f_pen(x, mu):
    return x**2 + mu * max(0, 1 - x)**2

mu_values = [1, 10, 100, 1000, 10000]
results = []
for mu in mu_values:
    res = minimize_scalar(lambda x: f_pen(x, mu), bounds=(-5, 5), method='bounded')
    results.append((mu, res.x, max(0, 1-res.x)))

header('Exercise 5: Penalty Method')
print(f"{'mu':>8} | {'x(mu)':>10} | {'Violation':>10}")
print('-' * 35)
for mu, x_mu, viol in results:
    print(f'{mu:>8} | {x_mu:>10.6f} | {viol:>10.6f}')

# Analytical: x(mu) = mu/(1+mu)
for mu, x_mu, _ in results:
    x_analytical = mu / (1 + mu)
    if not np.isclose(x_mu, x_analytical, atol=1e-4):
        print(f'WARNING: mu={mu}, numerical={x_mu:.6f}, analytical={x_analytical:.6f}')

check_close('x(mu) -> 1 as mu -> inf', results[-1][1], 1.0, tol=1e-3)
check_true('violation decreases', results[-1][2] < results[0][2])

print('\nTakeaway: Penalty methods converge to the constrained solution as mu -> infinity.')

---

### Exercise 6: ADMM for Lasso ★★

Implement ADMM for the Lasso problem.

**(a)** Formulate Lasso as $\min \frac{1}{2}\lVert A\mathbf{x} - \mathbf{b} \rVert_2^2 + \lambda \lVert \mathbf{z} \rVert_1$ s.t. $\mathbf{x} = \mathbf{z}$.

**(b)** Derive the three ADMM update steps.

**(c)** Implement and verify convergence.

In [ ]:
# Exercise 6: Your Solution
def admm_lasso(A, b, lam, rho=1.0, max_iter=500, tol=1e-6):
    n, p = A.shape
    x = np.zeros(p)
    z = np.zeros(p)
    u = np.zeros(p)
    L = np.linalg.cholesky(A.T @ A + rho * np.eye(p))
    for k in range(max_iter):
        rhs = A.T @ b + rho * (z - u)
        x = np.linalg.solve(L.T, np.linalg.solve(L, rhs))
        z_old = z.copy()
        z = soft_threshold(x + u, lam / rho)
        u = u + x - z
        if np.linalg.norm(x - z) < tol and rho * np.linalg.norm(z - z_old) < tol:
            break
    return x, k + 1

np.random.seed(42)
m, p = 100, 50
A = np.random.randn(m, p)
x_true = np.zeros(p)
x_true[:5] = [1.0, -0.5, 0.8, -0.3, 0.6]
b = A @ x_true + 0.1 * np.random.randn(m)

x_admm, n_iter = admm_lasso(A, b, lam=0.5)
print(f'ADMM Lasso: {n_iter} iterations')
print(f'Non-zeros: {np.sum(np.abs(x_admm) > 1e-6)}')
print(f'||x_admm - x_true||: {np.linalg.norm(x_admm - x_true):.6f}')

In [ ]:
# Exercise 6: Solution
def admm_lasso(A, b, lam, rho=1.0, max_iter=500, tol=1e-6):
    n, p = A.shape
    x = np.zeros(p)
    z = np.zeros(p)
    u = np.zeros(p)
    L = np.linalg.cholesky(A.T @ A + rho * np.eye(p))
    for k in range(max_iter):
        rhs = A.T @ b + rho * (z - u)
        x = np.linalg.solve(L.T, np.linalg.solve(L, rhs))
        z_old = z.copy()
        z = soft_threshold(x + u, lam / rho)
        u = u + x - z
        if np.linalg.norm(x - z) < tol and rho * np.linalg.norm(z - z_old) < tol:
            break
    return x, z, k + 1

np.random.seed(42)
m, p = 100, 50
A = np.random.randn(m, p)
x_true = np.zeros(p)
x_true[:5] = [1.0, -0.5, 0.8, -0.3, 0.6]
b = A @ x_true + 0.1 * np.random.randn(m)

x_admm, z_admm, n_iter = admm_lasso(A, b, lam=0.5)
active = np.sum(np.abs(z_admm) > 0.1)

header('Exercise 6: ADMM for Lasso')
print(f'Iterations: {n_iter}')
print(f'Active coefficients (>|0.1|): {active} (true: 5)')
print(f'||z_admm - x_true||: {np.linalg.norm(z_admm - x_true):.6f}')

check_true('terminated within the allotted iterations', n_iter <= 500)
check_true('thresholded solution is sparse', active <= 10)
check_true('error is small', np.linalg.norm(z_admm - x_true) < 0.2)

print()
print('Takeaway: ADMM splits Lasso into a least-squares step and a soft-thresholding step; sparsity is reflected in the thresholded z variable.')

---

### Exercise 7: Portfolio Optimization ★★★

Solve mean-variance portfolio optimization with non-negativity constraints.

**(a)** Formulate: $\min \frac{1}{2}\mathbf{w}^\top \Sigma \mathbf{w}$ s.t. $\boldsymbol{\mu}^\top \mathbf{w} = r_{\text{target}}$, $\mathbf{1}^\top \mathbf{w} = 1$, $\mathbf{w} \geq \mathbf{0}$.

**(b)** Solve using projected gradient descent.

**(c)** Compare with the unconstrained solution.

In [ ]:
# Exercise 7: Your Solution
np.random.seed(42)
n_assets = 10
mu = np.random.uniform(0.05, 0.15, n_assets)
Sigma = np.random.randn(n_assets, n_assets)
Sigma = Sigma.T @ Sigma / n_assets + 0.01 * np.eye(n_assets)

def f_port(w): return 0.5 * w @ Sigma @ w
def grad_port(w): return Sigma @ w

def proj_constraints(w):
    w = np.maximum(w, 0)
    return proj_simplex(w)

w = np.ones(n_assets) / n_assets
L = np.linalg.norm(Sigma)
eta = 1.0 / L

for t in range(5000):
    w = proj_constraints(w - eta * grad_port(w))

print(f'Portfolio return: {w @ mu:.6f}')
print(f'Portfolio risk: {np.sqrt(w @ Sigma @ w):.6f}')
print(f'Min weight: {w.min():.6f}')
print(f'Sum of weights: {w.sum():.6f}')

In [ ]:
# Exercise 7: Solution
np.random.seed(42)
n_assets = 10
mu = np.random.uniform(0.05, 0.15, n_assets)
Sigma = np.random.randn(n_assets, n_assets)
Sigma = Sigma.T @ Sigma / n_assets + 0.01 * np.eye(n_assets)

def f_port(w): return 0.5 * w @ Sigma @ w
def grad_port(w): return Sigma @ w

def proj_constraints(w):
    w = np.maximum(w, 0)
    return proj_simplex(w)

w = np.ones(n_assets) / n_assets
L = np.linalg.norm(Sigma)
eta = 1.0 / L

for t in range(5000):
    w = proj_constraints(w - eta * grad_port(w))

header('Exercise 7: Portfolio Optimization')
print(f'Portfolio return: {w @ mu:.6f}')
print(f'Portfolio risk: {np.sqrt(w @ Sigma @ w):.6f}')
print(f'Min weight: {w.min():.6f}')
print(f'Sum of weights: {w.sum():.6f}')

check_true('non-negative weights', w.min() >= -1e-10)
check_close('weights sum to 1', w.sum(), 1.0)
check_true('risk is finite', np.sqrt(w @ Sigma @ w) < 1.0)

print('\nTakeaway: Constrained portfolio optimization ensures realistic, implementable portfolios.')

---

### Exercise 8: Constrained RL Lagrangian Analysis ★★★

Analyze the Lagrangian formulation of a constrained MDP.

**(a)** For a 2-state, 2-action CMDP with reward $r$ and cost $c$, write the Lagrangian $\mathcal{L}(\pi, \lambda)$.

**(b)** Show that optimizing $\mathcal{L}$ for fixed $\lambda$ is an unconstrained MDP with modified reward $r' = r - \lambda c$.

**(c)** Analyze how varying $\lambda$ traces the Pareto frontier between reward and cost.

In [ ]:
# Exercise 8: Your Solution
R = np.array([[1.0, 0.5], [0.8, 0.3]])
C = np.array([[0.1, 0.0], [0.3, 0.1]])
d_max = 0.15

def expected_reward_cost(pi):
    r = 0.5*(pi[0]*R[0,0]+(1-pi[0])*R[0,1]+pi[1]*R[1,0]+(1-pi[1])*R[1,1])
    c = 0.5*(pi[0]*C[0,0]+(1-pi[0])*C[0,1]+pi[1]*C[1,0]+(1-pi[1])*C[1,1])
    return r, c

lambda_values = np.linspace(0, 10, 50)
reward_vals, cost_vals = [], []

for lam in lambda_values:
    R_mod = R - lam * C
    pi = np.array([1.0 if R_mod[s,0]>=R_mod[s,1] else 0.0 for s in range(2)])
    r, c = expected_reward_cost(pi)
    reward_vals.append(r)
    cost_vals.append(c)

reward_vals = np.array(reward_vals)
cost_vals = np.array(cost_vals)

print(f'Max reward (lambda=0): r={reward_vals[0]:.4f}, c={cost_vals[0]:.4f}')
print(f'Min cost (lambda=10): r={reward_vals[-1]:.4f}, c={cost_vals[-1]:.4f}')
feasible = cost_vals <= d_max
if np.any(feasible):
    best_idx = np.argmax(reward_vals[feasible])
    print(f'Best feasible: r={reward_vals[feasible][best_idx]:.4f}, c={cost_vals[feasible][best_idx]:.4f}')
else:
    print('No feasible policy found')

In [ ]:
# Exercise 8: Solution
R = np.array([[1.0, 0.5], [0.8, 0.3]])
C = np.array([[0.1, 0.0], [0.3, 0.1]])
d_max = 0.15

def expected_reward_cost(pi):
    r = 0.5*(pi[0]*R[0,0]+(1-pi[0])*R[0,1]+pi[1]*R[1,0]+(1-pi[1])*R[1,1])
    c = 0.5*(pi[0]*C[0,0]+(1-pi[0])*C[0,1]+pi[1]*C[1,0]+(1-pi[1])*C[1,1])
    return r, c

lambda_values = np.linspace(0, 10, 50)
reward_vals, cost_vals = [], []

for lam in lambda_values:
    R_mod = R - lam * C
    pi = np.array([1.0 if R_mod[s,0]>=R_mod[s,1] else 0.0 for s in range(2)])
    r, c = expected_reward_cost(pi)
    reward_vals.append(r)
    cost_vals.append(c)

reward_vals = np.array(reward_vals)
cost_vals = np.array(cost_vals)

header('Exercise 8: Constrained RL')
print(f'Max reward (lambda=0): r={reward_vals[0]:.4f}, c={cost_vals[0]:.4f}')
print(f'Min cost (lambda=10): r={reward_vals[-1]:.4f}, c={cost_vals[-1]:.4f}')

feasible = cost_vals <= d_max
if np.any(feasible):
    best_idx = np.argmax(reward_vals[feasible])
    print(f'Best feasible: r={reward_vals[feasible][best_idx]:.4f}, c={cost_vals[feasible][best_idx]:.4f}')

check_true('Pareto frontier is non-increasing', np.all(np.diff(reward_vals) <= 1e-10))
check_true('cost decreases with lambda', cost_vals[-1] < cost_vals[0])

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(cost_vals, reward_vals, 'o-', color='#0077BB')
    ax.axvline(d_max, color='#CC3311', linestyle='--', label=f'Cost limit={d_max}')
    feasible = cost_vals <= d_max
    if np.any(feasible):
        best_idx = np.argmax(reward_vals[feasible])
        ax.plot(cost_vals[feasible][best_idx], reward_vals[feasible][best_idx],
               's', color='#EE3377', markersize=10, label='Best feasible')
    ax.set_title('Pareto frontier: Reward vs Cost')
    ax.set_xlabel('Expected Cost')
    ax.set_ylabel('Expected Reward')
    ax.legend()
    fig.tight_layout()
    plt.show()

print('\nTakeaway: The Lagrange multiplier lambda controls the reward-cost tradeoff in constrained RL.')

---

## What to Review After Finishing

- [ ] Exercise 1: Lagrange multipliers balance objective and constraint gradients
- [ ] Exercise 2: KKT conditions systematically identify constrained optima
- [ ] Exercise 3: Simplex projection via sorting-based O(n log n) algorithm
- [ ] Exercise 4: SVM dual reveals kernel trick and support vector sparsity
- [ ] Exercise 5: Penalty methods converge as mu -> infinity
- [ ] Exercise 6: ADMM splits Lasso into least-squares + soft-thresholding
- [ ] Exercise 7: Constrained portfolio optimization with non-negativity
- [ ] Exercise 8: Lagrangian formulation of constrained RL

## References

1. Boyd, S. & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.
2. Nocedal, J. & Wright, S. (2006). *Numerical Optimization* (2nd ed.). Springer.
3. Bertsekas, D. (2014). *Constrained Optimization and Lagrange Multiplier Methods*. Athena Scientific.